Ghaniyul Amri Caulava
AI25

import libraries

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("seaborn-v0_8")

pd.set_option("display.max_columns", None)
pd.set_option("display.precision", 3)

Load Dataset

In [ ]:
df = sns.load_dataset("mpg")

print("First five rows")
display(df.head())

Stage 1
Task 1
Inspection & Data Cleaning

In [ ]:
print("Shape of dataset:")
print(df.shape)

print("\n")

print("Data Types")
display(df.dtypes)

print("\nMissing Values")
display(df.isnull().sum())

Interpretation

The dataset consists of vehicle specifications and fuel efficiency measurements.

Before performing any statistical analysis, it is important to inspect the dataset for missing values because incomplete data may bias the results or cause errors during numerical computations.

From the inspection above, only the horsepower column contains missing values, while the remaining variables are complete.

In [ ]:
missing_percent = df.isnull().mean() * 100

display(
    pd.DataFrame({
        "Missing Count": df.isnull().sum(),
        "Missing (%)": missing_percent
    })
)

Data Cleaning Strategy

The only feature containing missing values is horsepower.

Since horsepower is one of the most important numerical variables used throughout this analysis, imputing the missing values may distort the relationship between horsepower and fuel efficiency.

Furthermore, the dataset only contains a very small number of missing observations (less than 2% of all rows). Therefore, removing those rows is unlikely to affect the overall statistical properties of the dataset while preserving data integrity.

For this reason, row deletion (dropna) is chosen instead of mean or median imputation.

In [ ]:
df = df.dropna().reset_index(drop=True)

print("Dataset after cleaning")

print(df.shape)

print()

print(df.isnull().sum())

Stage 1 Task 2 SUmmary Statictics Using Numpy Only

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns

numeric_cols

In [ ]:
summary = []

for col in numeric_cols:

    arr = df[col].to_numpy()

    mean = np.mean(arr)
    median = np.median(arr)
    std = np.std(arr)

    q1 = np.percentile(arr,25)
    q3 = np.percentile(arr,75)

    iqr = q3-q1

    lower = q1 - 1.5*iqr
    upper = q3 + 1.5*iqr

    outliers = np.sum((arr<lower) | (arr>upper))

    summary.append([
        col,
        mean,
        median,
        std,
        iqr,
        outliers
    ])

summary_df = pd.DataFrame(
    summary,
    columns=[
        "Feature",
        "Mean",
        "Median",
        "Std",
        "IQR",
        "Outlier Count"
    ]
)

summary_df

Interpretation

The summary statistics provide an overview of each numerical feature.

The mean measures the average value.
The median is more robust against outliers.
The standard deviation describes variability.
The interquartile range (IQR) measures the spread of the middle 50% of observations.
Outliers are detected using the 1.5 × IQR rule, which is less sensitive to extreme values than methods based solely on the mean and standard deviation.

Comparing the mean and median can also provide insight into whether a feature is approximately symmetric or skewed.

Stage 2 Task 3 Extract Numeric Features & Z-Score Standardization

Objective

In machine learning, features often have different scales. For example, vehicle weight is measured in thousands of pounds, while MPG is measured in tens. To ensure that each feature contributes equally during analysis or model training, we standardize the data using the Z-score formula:

z=
(x−μ)/σ
	​


This task requires implementing the standardization manually using NumPy only, without using sklearn or loops.

In [ ]:
features = [
    "mpg",
    "displacement",
    "horsepower",
    "weight",
    "acceleration"
]

X = df[features].to_numpy()

print("Shape:", X.shape)
print()
print(X[:5])

In [ ]:
mean = np.mean(X, axis=0)
std = np.std(X, axis=0)

X_z = (X - mean) / std

print("Mean after standardization:")
print(np.round(X_z.mean(axis=0), 5))

print()

print("Standard deviation after standardization:")
print(np.round(X_z.std(axis=0), 5))

Interpretation

The standardized dataset has:

Mean ≈ 0
Standard deviation ≈ 1

This confirms that the Z-score standardization was successfully implemented using vectorized NumPy operations.

Standardizing features is especially important before applying machine learning algorithms or dimensionality reduction techniques such as PCA.

Stage 2
Task 4 — Correlation Matrix using NumPy

Objective

Correlation measures the strength and direction of the linear relationship between variables.

The Pearson correlation coefficient ranges from −1 to +1:

+1 → Perfect positive correlation
−1 → Perfect negative correlation
0 → No linear relationship

High correlations between predictor variables may indicate multicollinearity, which can negatively affect some machine learning models.

In [ ]:
corr_matrix = np.corrcoef(X, rowvar=False)

corr_df = pd.DataFrame(
    corr_matrix,
    index=features,
    columns=features
)

display(corr_df.round(3))

In [ ]:
corr_copy = corr_matrix.copy()
np.fill_diagonal(corr_copy, np.nan)
upper = np.triu(corr_copy, k=1)

positive_index = np.nanargmax(upper)

i, j = np.unravel_index(positive_index, upper.shape)

print("Strongest Positive Correlation")
print(f"{features[i]} <-> {features[j]}")
print(f"Correlation = {corr_matrix[i,j]:.3f}")

print()

mpg_corr = corr_matrix[0,1:]

lowest = np.argmin(mpg_corr)

print("Strongest Negative Correlation with MPG")

print(f"mpg <-> {features[lowest+1]}")
print(f"Correlation = {mpg_corr[lowest]:.3f}")

print()
input_corr = corr_matrix[1:,1:].copy()

np.fill_diagonal(input_corr, np.nan)

idx = np.nanargmax(np.abs(input_corr))

r, c = np.unravel_index(idx, input_corr.shape)

print("Most Correlated Input Features")

print(f"{features[r+1]} <-> {features[c+1]}")
print(f"Correlation = {input_corr[r,c]:.3f}")

Interpretation

The correlation matrix reveals several important relationships:

The strongest positive correlation indicates two variables that tend to increase together.
The strongest negative correlation with MPG identifies the feature that is most strongly associated with lower fuel efficiency.
A very high correlation between two predictor variables suggests multicollinearity, meaning both variables contain similar information and may be redundant in predictive models.

Stage 2
Task 5 — Boolean Masking Analysis

Objective

Boolean masking enables efficient filtering of NumPy arrays without using loops.

In this task, we investigate whether cars with above-average horsepower also tend to have above-average weight.

In [ ]:
horsepower = df["horsepower"].to_numpy()
weight = df["weight"].to_numpy()

mean_hp = horsepower.mean()
mean_weight = weight.mean()

mask = horsepower > mean_hp

heavy_hp_weight = weight[mask].mean()

difference = heavy_hp_weight - mean_weight

print(f"Average horsepower : {mean_hp:.2f}")
print(f"Average weight      : {mean_weight:.2f}")
print(f"Average weight (High HP Cars): {heavy_hp_weight:.2f}")

print()

print(f"Absolute Difference : {abs(difference):.2f} lbs")

Interpretation

The analysis compares the average weight of vehicles with above-average horsepower to the overall dataset average.

If the calculated difference is positive, it indicates that high-horsepower vehicles generally weigh more than the average vehicle.

This finding is expected because larger engines capable of producing higher horsepower are typically installed in heavier vehicles. Consequently, increased vehicle weight contributes to lower fuel efficiency, reinforcing the negative relationship between weight and MPG observed in the correlation analysis.

Stage 3 – Data Visualization
Task 6 – MPG Distribution

MPG Distribution

This histogram visualizes the distribution of vehicle fuel efficiency (MPG). The plot includes:

Histogram
KDE (Kernel Density Estimation)
Mean line
Median line
Value annotations

This visualization helps determine whether the distribution is approximately symmetric or skewed.

In [ ]:
plt.figure(figsize=(10,6))

sns.histplot(
    data=df,
    x="mpg",
    bins=25,
    kde=True,
    color="steelblue"
)

mean_mpg = df["mpg"].mean()
median_mpg = df["mpg"].median()

plt.axvline(
    mean_mpg,
    color="red",
    linestyle="--",
    linewidth=2,
    label=f"Mean = {mean_mpg:.2f}"
)

plt.axvline(
    median_mpg,
    color="green",
    linestyle="--",
    linewidth=2,
    label=f"Median = {median_mpg:.2f}"
)

plt.text(mean_mpg+0.3,20,f"{mean_mpg:.2f}",color="red")
plt.text(median_mpg+0.3,18,f"{median_mpg:.2f}",color="green")

plt.title("Distribution of Vehicle Fuel Efficiency (MPG)")
plt.xlabel("Miles per Gallon")
plt.ylabel("Frequency")
plt.legend()

plt.show()

Interpretation

The histogram shows that the MPG values are slightly right-skewed.

The mean is slightly higher than the median, indicating that a number of highly fuel-efficient vehicles create a longer tail toward higher MPG values.

Overall, most vehicles cluster around the middle MPG range, while only a few achieve exceptionally high fuel efficiency.

Fuel Efficiency by Country of Origin

This boxplot compares the MPG distributions of vehicles manufactured in:

USA
Europe
Japan

The chart evaluates both average fuel efficiency and variability across regions.

In [ ]:
plt.figure(figsize=(9,6))

sns.boxplot(
    data=df,
    x="origin",
    y="mpg",
    palette="Set2",
    order=["usa","europe","japan"]
)

plt.title("Fuel Efficiency by Country of Origin")
plt.xlabel("Origin")
plt.ylabel("Miles per Gallon")

plt.show()

Interpretation

From the boxplot:

Japanese vehicles generally have the highest median MPG, indicating superior fuel efficiency.
European vehicles also perform well with relatively high MPG values.
American vehicles tend to have lower fuel efficiency and exhibit greater variation in MPG.

Japan also demonstrates a relatively compact interquartile range, suggesting more consistent fuel efficiency among its vehicles.

Task 8 – Weight vs MPG

Relationship Between Weight and Fuel Efficiency

This scatter plot investigates the relationship between vehicle weight and MPG.

Vehicle cylinders are represented by different colors, while a manually computed regression line (using np.polyfit) illustrates the overall trend.

In [ ]:
plt.figure(figsize=(10,6))

sns.scatterplot(
    data=df,
    x="weight",
    y="mpg",
    hue="cylinders",
    palette="viridis"
)

coef = np.polyfit(df["weight"], df["mpg"],1)

poly = np.poly1d(coef)

x_line = np.linspace(
    df["weight"].min(),
    df["weight"].max(),
    100
)

plt.plot(
    x_line,
    poly(x_line),
    color="red",
    linewidth=2,
    label="Trend Line"
)

corr = np.corrcoef(
    df["weight"],
    df["mpg"]
)[0,1]

plt.title(f"Weight vs MPG (Correlation = {corr:.3f})")

plt.xlabel("Weight (lbs)")
plt.ylabel("Miles per Gallon")

plt.legend()

plt.show()

Interpretation

The scatter plot reveals a strong negative linear relationship between vehicle weight and fuel efficiency.

As vehicle weight increases, MPG generally decreases.

The downward-sloping regression line and the large negative correlation coefficient confirm that heavier vehicles consume more fuel than lighter vehicles.

Task 9 – Correlation Heatmap

Correlation Heatmap

The heatmap summarizes the pairwise Pearson correlations among the five numerical features analyzed in Stage 2.

A diverging color palette is used to distinguish positive and negative correlations.

In [ ]:
plt.figure(figsize=(8,6))

sns.heatmap(
    corr_df,
    annot=True,
    cmap="coolwarm",
    center=0,
    fmt=".2f",
    square=True
)

plt.title("Correlation Matrix of Numerical Features")

plt.show()

Interpretation

Several important relationships can be observed from the heatmap:

MPG has a strong negative correlation with both weight and displacement, indicating that larger and heavier vehicles tend to be less fuel-efficient.
Weight and displacement exhibit a strong positive correlation, suggesting that larger engines are generally installed in heavier vehicles.
Horsepower also shows strong positive correlations with displacement and weight.

These high correlations among predictor variables indicate the presence of multicollinearity, meaning several variables provide overlapping information.

Stage 4 – Contextual Interpretation
Task 10 — What Factor Most Strongly Predicts Fuel Efficiency?

What Factor Most Strongly Predicts Vehicle Fuel Efficiency?

Based on the exploratory data analysis, vehicle weight appears to be the strongest predictor of fuel efficiency (MPG).

Several findings support this conclusion:

Strong Negative Correlation
The correlation matrix shows that weight has the strongest negative correlation with MPG, indicating that fuel efficiency decreases as vehicle weight increases.
This relationship is stronger than those observed for displacement, horsepower, or acceleration.
Scatter Plot Analysis
The scatter plot of Weight vs MPG shows a clear downward linear trend.
The regression line generated using np.polyfit() confirms that heavier vehicles consistently achieve lower MPG values.
Compared to other variables, weight exhibits the clearest linear relationship with fuel efficiency.
Boolean Masking Result
Vehicles with above-average horsepower also have an average weight greater than the dataset average.
This suggests that high-performance vehicles tend to be heavier, indirectly contributing to reduced fuel efficiency.
Correlation Heatmap
The heatmap indicates that weight is highly correlated with displacement and horsepower.
This multicollinearity suggests that these variables often increase together; however, weight remains the most directly associated with MPG.
Origin Comparison
The boxplot reveals that Japanese vehicles generally achieve higher MPG values than American vehicles.
One possible explanation is that Japanese manufacturers historically produced smaller and lighter vehicles, which naturally consume less fuel.

Overall Conclusion

Among all numerical variables analyzed, vehicle weight is the most influential factor affecting fuel efficiency.

Reducing vehicle weight is likely to have the greatest positive impact on improving MPG, making it an important design consideration for automotive engineers seeking to enhance fuel economy.

Task 11 – Mean MPG per Decade

Fuel Efficiency Trend by Decade

To understand how fuel efficiency has evolved over time, the manufacturing year is rounded to the nearest decade. The average MPG is then calculated for each decade and visualized using a line chart.

In [ ]:
df["decade"] = (df["model_year"] // 10) * 10
mpg_decade = (
    df.groupby("decade")["mpg"]
      .mean()
      .reset_index()
)

display(mpg_decade)

In [ ]:
plt.figure(figsize=(9,5))

sns.lineplot(
    data=mpg_decade,
    x="decade",
    y="mpg",
    marker="o",
    linewidth=2
)

plt.title("Average Fuel Efficiency by Decade")
plt.xlabel("Model Year (Decade)")
plt.ylabel("Average MPG")

plt.grid(alpha=0.3)

plt.show()

Interpretation

The line chart demonstrates a clear upward trend in average MPG over successive decades.

Several factors may explain this improvement:

Technological Advancements: Improvements in engine efficiency, fuel injection systems, and transmission technology have enabled vehicles to travel farther using the same amount of fuel.
Fuel Economy Regulations: The oil crises of the 1970s encouraged governments to introduce stricter fuel economy standards, prompting manufacturers to produce more efficient vehicles.
Lighter Vehicle Designs: Advances in materials engineering have reduced vehicle weight without compromising safety, contributing to better fuel efficiency.
Aerodynamic Improvements: Modern vehicle designs minimize air resistance, allowing engines to operate more efficiently at higher speeds.

Overall, the increasing MPG trend reflects continuous innovation in automotive engineering and a growing emphasis on energy efficiency and environmental sustainability.

Overall Conclusion

This analysis demonstrates that exploratory data analysis (EDA) is an essential first step before developing predictive models.

Key findings include:

The dataset contains only a small number of missing values, which were appropriately handled by removing incomplete rows.
Weight exhibits the strongest negative relationship with fuel efficiency, making it the most important predictor of MPG.
High horsepower vehicles are generally heavier, further reinforcing the relationship between vehicle mass and fuel consumption.
Japanese vehicles tend to have the highest fuel efficiency among the three regions analyzed.
Vehicle fuel efficiency has improved steadily over time, reflecting technological progress and stricter fuel economy standards.
Strong correlations among weight, displacement, and horsepower indicate multicollinearity, an important consideration when building regression models.

Overall, vehicle weight emerges as the dominant factor influencing fuel efficiency and should be prioritized in predictive modeling and engineering design decisions.